In [1]:
!pip install fastapi uvicorn pyngrok transformers==4.52.4 accelerate -q
!pip install faiss-cpu langchain langchain-community langchain-classic langchain-core langchain-huggingface pypdf sentence-transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 31.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.8 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import torch
import re
import json

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
).eval()

def generate_text(prompt, max_length=300):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [3]:
response_schemas = [
    ResponseSchema(
        name="full_name", 
        description="Candidate's full name", 
        type="string"
    ),
    ResponseSchema(
        name="email", 
        description="Candidate's email address", 
        type="string"
    ),
    ResponseSchema(
        name="education",
        description="List of educational qualifications, where each item is an object with 'degree', 'institution', and 'year' (as integer)",
        type="List[dict]",
    ),
    ResponseSchema(
        name="skills",
        description="List of candidate skills as strings",
        type="List[string]",
    ),
    ResponseSchema(
        name="experience",
        description="List of work experiences, where each item is an object with 'role', 'company', and 'years'",
        type="List[dict]",
    ),
]


output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

info_extraction_template = """
You are a smart assistant that extracts information from a user's cv.

Extract all information regarding the name of the user, his email, his education, his skills, and his experience.

Respond ONLY in JSON format as follows:
{format_instructions}

Example Input:
"Mohamed Negm - moahamed123@email.com
Education: B.Sc. Social Science, KFC, 2019
Skills: cooking, selling, cleaning
Experience:
- Worked at MC as a chef assistant (2020-2023)
- Chef at LHC (2023-Present)
"

Now extract from the following input:
"{user_input}"
"""

In [4]:
prompt = PromptTemplate(
    template=info_extraction_template,
    input_variables=["user_input"],
    partial_variables={"format_instructions": format_instructions},
)


In [ ]:
# Add the API key and ngrok token to the environment variables
NGROK_TOKEN = "NGROK_TOKEN"
API_KEY = "API_KEY"

In [10]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

app = FastAPI()

@app.post("/generate")
async def gen(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    data = await req.json()
    sample_input = data.get("prompt", "")
    formatted_prompt_str = prompt.format(user_input=sample_input)
    
    return {
        "response": generate_text(
            formatted_prompt_str,
            data.get("max_length", 1000)
        )
    }

In [11]:
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://detergent-direction-wrongness.ngrok-free.dev


INFO:     Started server process [105]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:36147 (Press CTRL+C to quit)


INFO:     197.36.70.249:0 - "POST /generate HTTP/1.1" 200 OK
